# World Cup Model Analysis

This notebook compares the performance of the candidate models used for World Cup prediction.

The following machine learning models were evaluated throughout the project:

- **Logistic Regression** – Linear probabilistic classifier used as a strong and interpretable baseline.
- **Random Forest** – Ensemble of decision trees trained using bootstrap aggregation (bagging).
- **XGBoost** – Gradient boosting model based on decision trees with regularization and advanced optimization techniques.
- **LightGBM** – Efficient gradient boosting framework designed for fast training and low memory usage.
- **CatBoost** – Gradient boosting algorithm that handles categorical features effectively and reduces prediction shift.

Model selection was performed using a two-stage evaluation process. First, a Leave-One-World-Cup-Out cross-validation was used to measure predictive performance through log loss. Then, the best feature and hyperparameter combinations were further evaluated using Monte Carlo World Cup simulations and the Tournament Loss metric, which measures how much probability each model assigns to the actual tournament champions.

In [2]:
import ast

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from core.constants import TOURNAMENT_EVAL_PATH, SELECTED_MODEL_RESULT, REAL_CHAMPIONS
from core.features import feats_from_str_list

## Load Evaluation Results

This section loads the evaluation results generated during the model selection process.

In [3]:
tournament_eval_df = pd.read_csv(TOURNAMENT_EVAL_PATH)
selected_model_df = pd.read_csv(SELECTED_MODEL_RESULT)
selected_model = selected_model_df.iloc[0]

## Tournament Evaluation Ranking

The table below ranks all evaluated models according to their Tournament Loss. Lower values indicate better tournament-level performance.

In [4]:
ranking = tournament_eval_df.sort_values("tournament_loss")
cols = [
    "model_name",
    "tournament_loss",
    "avg_loss",
    "hit_champion_count",
    "champion_in_first_four_count",
]
tournament_eval_df[cols]

,model_name,tournament_loss,avg_loss,hit_champion_count,champion_in_first_four_count
0,XGBoost,1.961305,0.953846,2,3
1,Random Forest,2.000797,0.946600,2,3
2,CatBoost,2.015141,0.940014,1,3
3,LightGBM,2.032098,0.961985,2,3
4,CatBoost,2.082426,0.941722,1,2
5,LightGBM,2.083472,0.949947,2,4
6,XGBoost,2.096419,0.946338,2,3
7,XGBoost,2.098760,0.954036,2,3
8,CatBoost,2.136958,0.937735,1,2
9,Random Forest,2.161577,0.945923,2,3


## Real Champion Probabilities

This table shows the probability assigned by the selected model to the actual World Cup champion in each tournament. Higher probabilities indicate that the model was more confident in predicting the eventual winner, while lower probabilities suggest that the champion was considered a less likely outcome by the model.

In [5]:
champ_probs = ast.literal_eval(selected_model["real_champion_probs"])

rows = []

for year, champion in REAL_CHAMPIONS.items():
    rows.append({
        "World Cup": year,
        "Real Champion": champion.title(),
        "Model Probability": champ_probs[year]
    })

champion_probs_df = pd.DataFrame(rows)

champion_probs_df

,World Cup,Real Champion,Model Probability
0,2006,Italy,0.04935
1,2010,Spain,0.33710
2,2014,Germany,0.19840
3,2018,France,0.09305
4,2022,Argentina,0.20165


## World Cup Podium Predictions

The tables below show the four teams with the highest championship probabilities predicted by the selected model for each World Cup.

In [6]:
pods = {
    2006: ast.literal_eval(selected_model["2006_pod"]),
    2010: ast.literal_eval(selected_model["2010_pod"]),
    2014: ast.literal_eval(selected_model["2014_pod"]),
    2018: ast.literal_eval(selected_model["2018_pod"]),
    2022: ast.literal_eval(selected_model["2022_pod"]),
}

def build_pod_df(year):
    pod = pods[year]
    champion = REAL_CHAMPIONS[year]

    rows = []

    for team, prob in pod.items():
        rows.append({
            "Team": team,
            "Probability": round(prob, 4),
            "Champion": "🏆 YES" if team == champion else ""
        })

    return pd.DataFrame(rows)

In [7]:
build_pod_df(2006)

,Team,Probability,Champion
0,france,0.1706,
1,england,0.1524,
2,brazil,0.1234,
3,argentina,0.1018,


In [8]:
build_pod_df(2010)

,Team,Probability,Champion
0,spain,0.3371,🏆 YES
1,france,0.1183,
2,england,0.1011,
3,brazil,0.0834,


In [9]:
build_pod_df(2014)

,Team,Probability,Champion
0,germany,0.1984,🏆 YES
1,spain,0.1978,
2,argentina,0.1299,
3,brazil,0.1206,


In [10]:
build_pod_df(2018)

,Team,Probability,Champion
0,brazil,0.2437,
1,germany,0.1673,
2,spain,0.1459,
3,france,0.0930,🏆 YES


In [11]:
build_pod_df(2022)

,Team,Probability,Champion
0,argentina,0.2016,🏆 YES
1,brazil,0.1787,
2,france,0.1361,
3,spain,0.1310,


## Selected Features

The selected model was trained using the features listed below.

In [12]:
best = tournament_eval_df.iloc[0]
feature_descriptions = {
    "att_diff": "Difference in attacking rating between both teams (EA Fifa game rating).",
    "mid_diff": "Difference in midfield rating between both teams (EA Fifa game rating).",
    "def_diff": "Difference in defensive rating between both teams (EA Fifa game rating).",
    "fifa_rank_diff": "Difference in FIFA ranking positions.",
    "titles_diff": "Difference in number of World Cup titles won.",
    "elo_diff": "Difference in Elo ratings between both teams."
}

features_str = ast.literal_eval(best["feats"])
feats = feats_from_str_list(features_str)


features_df = pd.DataFrame(
    [{"Feature": ft.label, "Description": ft.description} for ft in feats]
)

features_df.style.set_table_styles([
    {
        "selector": "th",
        "props": [("text-align", "left")]
    },
    {
        "selector": "td",
        "props": [
            ("white-space", "normal"),
            ("max-width", "600px")
        ]
    }
])



,Feature,Description
0,att_diff,Difference in attacking rating between both teams (EA Sports Fifa game rating).
1,mid_diff,Difference in midfield rating between both teams (EA Sports Fifa game rating).
2,def_diff,Difference in defensive rating between both teams (EA Sports Fifa game rating).
3,elo_rating_diff,Difference in Elo ratings between both teams.
4,titles_diff,Difference in number of World Cup titles won.
